# NHANES Diabetes Model Notebook

This notebook builds a diabetes decision-support model from NHANES laboratory data and prepares it to match your ontology.

## Files used

- `GHB_L.xpt` → Glycohemoglobin / HbA1c
- `GLU_L.xpt` → Fasting glucose
- `BMX_L.xpt` → Body measures / BMI
- `DEMO_L.xpt` → Demographics / age and gender

## Important medical note

This is a **decision-support model**, not a final medical diagnosis system.  
The target labels are created using clinical threshold rules so the ML model and ontology use the same medical logic.

In [ ]:
# If needed, install required packages:
# !pip install pandas scikit-learn matplotlib joblib pyreadstat

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

## 1. Load NHANES XPT files

All files are connected using the column `SEQN`, which is the participant ID.

In [ ]:
DATA_DIR = "."

ghb_path = os.path.join(DATA_DIR, "GHB_L.xpt")
glu_path = os.path.join(DATA_DIR, "GLU_L.xpt")
bmx_path = os.path.join(DATA_DIR, "BMX_L.xpt")
demo_path = os.path.join(DATA_DIR, "DEMO_L.xpt")

ghb = pd.read_sas(ghb_path)
glu = pd.read_sas(glu_path)
bmx = pd.read_sas(bmx_path)
demo = pd.read_sas(demo_path)

print("GHB:", ghb.shape)
print("GLU:", glu.shape)
print("BMX:", bmx.shape)
print("DEMO:", demo.shape)

## 2. Select columns needed for the Diabetes Panel

Important columns:

| NHANES column | Meaning | Ontology mapping |
|---|---|---|
| `LBXGH` | HbA1c (%) | `HbA1c_Test` |
| `LBXGLU` | Fasting Glucose (mg/dL) | `Fasting_Glucose_Test` |
| `BMXBMI` | BMI | `BMI_Measurement` |
| `RIDAGEYR` | Age | `Age_Factor` |
| `RIAGENDR` | Gender | `Gender_Factor` |

In [ ]:
ghb_small = ghb[["SEQN", "LBXGH"]].copy()
glu_small = glu[["SEQN", "LBXGLU"]].copy()
bmx_small = bmx[["SEQN", "BMXBMI"]].copy()
demo_small = demo[["SEQN", "RIDAGEYR", "RIAGENDR"]].copy()

df = ghb_small.merge(glu_small, on="SEQN", how="inner")
df = df.merge(bmx_small, on="SEQN", how="inner")
df = df.merge(demo_small, on="SEQN", how="inner")

df.head()

## 3. Rename columns for easier use

These names will also be easier to connect to your FastAPI backend and ontology.

In [ ]:
df = df.rename(columns={
    "LBXGH": "hba1c",
    "LBXGLU": "fasting_glucose",
    "BMXBMI": "bmi",
    "RIDAGEYR": "age",
    "RIAGENDR": "gender"
})

df.head()

## 4. Basic cleaning

We remove missing and impossible values only.

Do **not** remove medically abnormal values like high glucose or high HbA1c because these are exactly the disease cases the model needs to learn from.

In [ ]:
print("Before cleaning:", df.shape)
print(df.isna().sum())

df = df.dropna(subset=["hba1c", "fasting_glucose", "bmi", "age", "gender"])

# Keep medically possible values
df = df[
    (df["hba1c"].between(3.0, 18.0)) &
    (df["fasting_glucose"].between(40, 500)) &
    (df["bmi"].between(10, 80)) &
    (df["age"].between(12, 120))
].copy()

print("After cleaning:", df.shape)
df.describe()

## 5. Build the target label using clinical rules

Here we create the target using standard clinical thresholds:

| Class | Rule |
|---|---|
| `Healthy` | HbA1c < 5.7 and fasting glucose < 100 |
| `Prediabetes` | HbA1c 5.7–6.4 or fasting glucose 100–125 |
| `Diabetes` | HbA1c ≥ 6.5 or fasting glucose ≥ 126 |

This is useful because your ontology can use the same thresholds to produce findings like:

- `High_HbA1c`
- `High_Fasting_Glucose`
- `Prediabetes_Risk`
- `Diabetes_Candidate`

In [ ]:
def create_diabetes_target(row):
    hba1c = row["hba1c"]
    glucose = row["fasting_glucose"]

    if hba1c >= 6.5 or glucose >= 126:
        return "Diabetes"
    elif (5.7 <= hba1c < 6.5) or (100 <= glucose < 126):
        return "Prediabetes"
    else:
        return "Healthy"

df["target"] = df.apply(create_diabetes_target, axis=1)

df["target"].value_counts()

## 6. Create ontology findings from the same medical thresholds

This part helps match the ML model with the ontology.  
The ontology can use these findings as evidence.

In [ ]:
def get_ontology_findings(row):
    findings = []

    if row["hba1c"] >= 6.5:
        findings.append("High_HbA1c_Diabetes_Range")
    elif row["hba1c"] >= 5.7:
        findings.append("Elevated_HbA1c_Prediabetes_Range")
    else:
        findings.append("Normal_HbA1c")

    if row["fasting_glucose"] >= 126:
        findings.append("High_Fasting_Glucose_Diabetes_Range")
    elif row["fasting_glucose"] >= 100:
        findings.append("Elevated_Fasting_Glucose_Prediabetes_Range")
    else:
        findings.append("Normal_Fasting_Glucose")

    if row["bmi"] >= 30:
        findings.append("Obesity")
    elif row["bmi"] >= 25:
        findings.append("Overweight")
    else:
        findings.append("Normal_BMI")

    return findings

df["ontology_findings"] = df.apply(get_ontology_findings, axis=1)
df[["hba1c", "fasting_glucose", "bmi", "target", "ontology_findings"]].head()

## 7. Save the cleaned dataset

This file can be reused for training, testing, backend work, and documentation.

In [ ]:
df.to_csv("nhanes_diabetes_cleaned.csv", index=False)
print("Saved: nhanes_diabetes_cleaned.csv")

## 8. Train/Test Split

Important note:

Because the target is created from HbA1c and fasting glucose, using these same columns as features can cause **data leakage**.

So this notebook provides two options:

### Option A — Ontology-aligned demonstration model
Uses all clinical inputs including HbA1c and fasting glucose.  
This is good for prototype/demo but should be explained as rule-aligned.

### Option B — Less leakage model
Creates target mainly from HbA1c and trains using glucose + BMI + age + gender.  
This is better if the doctor asks about leakage.

We will train **Option B** as the safer academic version.

In [ ]:
# Safer feature set: do not use hba1c because it strongly defines the target.
# We keep fasting_glucose because it is a real diabetes lab input, but explain clearly that it also contributes to clinical labeling.
features = ["fasting_glucose", "bmi", "age", "gender"]
target = "target"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print(y_train.value_counts())

## 9. Preprocessing Pipeline

We use:

- Median imputation for numeric missing values
- Scaling for numeric features
- Most frequent imputation + one-hot encoding for gender

In [ ]:
numeric_features = ["fasting_glucose", "bmi", "age"]
categorical_features = ["gender"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 10. Train Random Forest Model

Random Forest is a good starting model for medical tabular data because it handles non-linear relationships and gives feature importance.

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        max_depth=None
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot()
plt.xticks(rotation=30)
plt.title("NHANES Diabetes Model - Confusion Matrix")
plt.show()

## 12. Cross Validation

In [ ]:
scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print("CV scores:", scores)
print("Mean accuracy:", scores.mean())
print("Std:", scores.std())

## 13. Feature Importance

In [ ]:
rf = model.named_steps["classifier"]

# Get transformed feature names
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
importances = rf.feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

importance_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(importance_df["feature"], importance_df["importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance")
plt.gca().invert_yaxis()
plt.show()

## 14. Prediction Function for Backend / FastAPI

This function returns:

- model prediction
- probabilities
- ontology findings
- recommended confirmatory tests
- recommended specialist

In [ ]:
def predict_diabetes_case(fasting_glucose, bmi, age, gender, hba1c=None):
    input_df = pd.DataFrame([{
        "fasting_glucose": fasting_glucose,
        "bmi": bmi,
        "age": age,
        "gender": gender
    }])

    prediction = model.predict(input_df)[0]
    probabilities = model.predict_proba(input_df)[0]

    probability_dict = {
        class_name: round(float(prob) * 100, 2)
        for class_name, prob in zip(model.classes_, probabilities)
    }

    # Ontology findings can still use HbA1c if the user enters it in the UI.
    findings_row = {
        "hba1c": hba1c if hba1c is not None else np.nan,
        "fasting_glucose": fasting_glucose,
        "bmi": bmi
    }

    findings = []

    if hba1c is not None:
        if hba1c >= 6.5:
            findings.append("High_HbA1c_Diabetes_Range")
        elif hba1c >= 5.7:
            findings.append("Elevated_HbA1c_Prediabetes_Range")
        else:
            findings.append("Normal_HbA1c")

    if fasting_glucose >= 126:
        findings.append("High_Fasting_Glucose_Diabetes_Range")
    elif fasting_glucose >= 100:
        findings.append("Elevated_Fasting_Glucose_Prediabetes_Range")
    else:
        findings.append("Normal_Fasting_Glucose")

    if bmi >= 30:
        findings.append("Obesity")
    elif bmi >= 25:
        findings.append("Overweight")
    else:
        findings.append("Normal_BMI")

    recommended_tests = [
        "Repeat HbA1c",
        "Repeat fasting plasma glucose",
        "Oral glucose tolerance test",
        "Urine microalbumin",
        "Lipid profile"
    ]

    return {
        "prediction": prediction,
        "probabilities_percent": probability_dict,
        "ontology_findings": findings,
        "recommended_tests": recommended_tests,
        "recommended_specialist": "Endocrinologist",
        "medical_note": "Decision support only. Not a final medical diagnosis."
    }

predict_diabetes_case(
    fasting_glucose=140,
    bmi=32,
    age=55,
    gender=2,
    hba1c=7.1
)

## 15. Save Model Artifacts

These files can be loaded in FastAPI.

In [ ]:
joblib.dump(model, "nhanes_diabetes_model.joblib")
joblib.dump(features, "nhanes_diabetes_features.joblib")

ontology_mapping = {
    "hba1c": "HbA1c_Test",
    "fasting_glucose": "Fasting_Glucose_Test",
    "bmi": "BMI_Measurement",
    "age": "Age_Factor",
    "gender": "Gender_Factor",
    "target_classes": ["Healthy", "Prediabetes", "Diabetes"],
    "findings": [
        "Normal_HbA1c",
        "Elevated_HbA1c_Prediabetes_Range",
        "High_HbA1c_Diabetes_Range",
        "Normal_Fasting_Glucose",
        "Elevated_Fasting_Glucose_Prediabetes_Range",
        "High_Fasting_Glucose_Diabetes_Range",
        "Normal_BMI",
        "Overweight",
        "Obesity"
    ]
}

import json
with open("diabetes_ontology_mapping.json", "w") as f:
    json.dump(ontology_mapping, f, indent=4)

print("Saved:")
print("- nhanes_diabetes_model.joblib")
print("- nhanes_diabetes_features.joblib")
print("- diabetes_ontology_mapping.json")

# What to tell the doctor

You can say:

> I used NHANES laboratory data and selected the files related to the Diabetes Panel.  
> I merged them using the participant ID `SEQN`.  
> Then I created the target labels using clinical threshold rules for HbA1c and fasting glucose, so the ontology and the ML model are aligned with the same medical logic.  
> The system is a decision-support tool, not a final diagnosis tool.

If the doctor asks about target labeling:

> The target was not invented manually. It was generated using standard clinical thresholds. This is called rule-based labeling and it is common when public datasets contain lab results but do not contain the exact diagnosis label required for the ML task.